In [1]:
using Julog, Test

In [3]:
num_var_cl = @julog [
    is_num(10) <<= true,
    # is_num(20) <<= true,

    is_var("a") <<= true,
    # is_var("b") <<= true,
]

2-element Vector{Clause}:
 is_num(10)
 is_var(a)

In [4]:
is_exp_cl = @julog [
    is_exp(n(I)) <<= is_num(I),
    is_exp(v(X)) <<= is_var(X),
    is_exp(add(E1, E2)) <<= is_exp(E1) & is_exp(E2),
    is_exp(let_(X, E1, E2)) <<= is_var(X) & is_exp(E1) & is_exp(E2),
]


4-element Vector{Clause}:
 is_exp(n(I)) <<= is_num(I)
 is_exp(v(X)) <<= is_var(X)
 is_exp(add(E1, E2)) <<= is_exp(E1) & is_exp(E2)
 is_exp(let_(X, E1, E2)) <<= is_var(X) & is_exp(E1) & is_exp(E2)

In [5]:
e1 = @julog (n(10))
e2 = @julog (add(n(10), n(10)))
e3 = @julog (add(n(10), add(n(10), n(10))))
e4 = @julog (let_("a", n(10), v("a")))
e5 = @julog (let_("a", n(10), add(v("a"), v("a"))))

let_(a, n(10), add(v(a), v(a)))

In [6]:
function check_is_exp(e)
    sat, subst = resolve(@julog(is_exp(:e)), [num_var_cl; is_exp_cl])
    @test sat == true
end

check_is_exp (generic function with 1 method)

In [7]:
check_is_exp(e1)
check_is_exp(e2)
check_is_exp(e3)
check_is_exp(e4)
check_is_exp(e5)

Test Passed

In [8]:
demand_cl = @julog [
    demand_exp(N1) <<= demand_exp(N) & (N > 0) & is(N1, N - 1)
]

1-element Vector{Clause}:
 demand_exp(N1) <<= demand_exp(N) & >(N, 0) & is(N1, -(N, 1))

In [9]:
demand_5 = @julog [demand_exp(5) <<= true]
demand_facts = derivations([demand_cl; demand_5], Inf)
# println(demand_facts)

6-element Vector{Clause}:
 demand_exp(5)
 demand_exp(4)
 demand_exp(3)
 demand_exp(2)
 demand_exp(1)
 demand_exp(0)

In [10]:
gen_exp_cl = @julog [
    gen_exp(1, n(I)) <<= is_num(I),
    gen_exp(1, v(X)) <<= is_var(X),
    # gen_exp(N, E) <<=
    #     gen_exp(N1, E1) & gen_exp(N2, E2) & is(N, N1 + N2 + 1) & is(E, add(E1, E2)),
    # gen_exp(N, E) <<=
    #     is_var(X) & gen_exp(N1, E1) & gen_exp(N2, E2) & is(N, N1 + N2 + 1) & is(E, let_(X, E1, E2)),
    gen_exp(N, add(E1, E2)) <<= demand_exp(N) &
        gen_exp(N1, E1) & gen_exp(N2, E2) & (N == N1 + N2 + 1),
    gen_exp(N, let_(X, E1, E2)) <<= demand_exp(N) &
        is_var(X) & gen_exp(N1, E1) & gen_exp(N2, E2) & (N == N1 + N2 + 1),
]


4-element Vector{Clause}:
 gen_exp(1, n(I)) <<= is_num(I)
 gen_exp(1, v(X)) <<= is_var(X)
 gen_exp(N, add(E1, E2)) <<= demand_exp(N) & gen_exp(N1, E1) & gen_exp(N2, E2) & ==(N, +(N1, N2, 1))
 gen_exp(N, let_(X, E1, E2)) <<= demand_exp(N) & is_var(X) & gen_exp(N1, E1) & gen_exp(N2, E2) & ==(N, +(N1, N2, 1))

In [11]:
exp_facts = derivations([num_var_cl; gen_exp_cl; demand_facts], Inf)
println(exp_facts)

Clause[gen_exp(1, n(10)), gen_exp(1, v(a)), gen_exp(3, add(n(10), n(10))), gen_exp(3, add(n(10), v(a))), gen_exp(3, add(v(a), n(10))), gen_exp(3, add(v(a), v(a))), gen_exp(3, let_(a, n(10), n(10))), gen_exp(3, let_(a, n(10), v(a))), gen_exp(3, let_(a, v(a), n(10))), gen_exp(3, let_(a, v(a), v(a))), gen_exp(5, add(n(10), add(n(10), n(10)))), gen_exp(5, add(n(10), add(n(10), v(a)))), gen_exp(5, add(n(10), add(v(a), n(10)))), gen_exp(5, add(n(10), add(v(a), v(a)))), gen_exp(5, add(n(10), let_(a, n(10), n(10)))), gen_exp(5, add(n(10), let_(a, n(10), v(a)))), gen_exp(5, add(n(10), let_(a, v(a), n(10)))), gen_exp(5, add(n(10), let_(a, v(a), v(a)))), gen_exp(5, add(v(a), add(n(10), n(10)))), gen_exp(5, add(v(a), add(n(10), v(a)))), gen_exp(5, add(v(a), add(v(a), n(10)))), gen_exp(5, add(v(a), add(v(a), v(a)))), gen_exp(5, add(v(a), let_(a, n(10), n(10)))), gen_exp(5, add(v(a), let_(a, n(10), v(a)))), gen_exp(5, add(v(a), let_(a, v(a), n(10)))), gen_exp(5, add(v(a), let_(a, v(a), v(a)))), gen_

In [12]:
exp_facts2 = derive(@julog[gen_exp(5, E)], [num_var_cl; gen_exp_cl; demand_facts])


(true, Dict{Var, Term}[{E => add(n(10), add(n(10), n(10)))}, {E => add(n(10), add(n(10), v(a)))}, {E => add(n(10), add(v(a), n(10)))}, {E => add(n(10), add(v(a), v(a)))}, {E => add(n(10), let_(a, n(10), n(10)))}, {E => add(n(10), let_(a, n(10), v(a)))}, {E => add(n(10), let_(a, v(a), n(10)))}, {E => add(n(10), let_(a, v(a), v(a)))}, {E => add(v(a), add(n(10), n(10)))}, {E => add(v(a), add(n(10), v(a)))}  …  {E => let_(a, add(v(a), v(a)), n(10))}, {E => let_(a, add(v(a), v(a)), v(a))}, {E => let_(a, let_(a, n(10), n(10)), n(10))}, {E => let_(a, let_(a, n(10), n(10)), v(a))}, {E => let_(a, let_(a, n(10), v(a)), n(10))}, {E => let_(a, let_(a, n(10), v(a)), v(a))}, {E => let_(a, let_(a, v(a), n(10)), n(10))}, {E => let_(a, let_(a, v(a), n(10)), v(a))}, {E => let_(a, let_(a, v(a), v(a)), n(10))}, {E => let_(a, let_(a, v(a), v(a)), v(a))}])